In [1]:
from datetime import datetime, timedelta, timezone
from typing import List, Optional
import feedparser
from pydantic import BaseModel
from docling.document_converter import DocumentConverter


/home/rym/Documents/Study/IA/LLM_PROJECT/venvnews/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class AnthropicArticle(BaseModel):
    title: str
    description: str
    url: str
    guid: str
    published_at: datetime
    category: Optional[str]= None

In [5]:
class AnthropicScrapper:
    def __init__(self):
        self.rss_urls=[
            "https://raw.githubusercontent.com/Olshansk/rss-feeds/main/feeds/feed_anthropic_news.xml",
            "https://raw.githubusercontent.com/Olshansk/rss-feeds/main/feeds/feed_anthropic_research.xml",
            "https://raw.githubusercontent.com/Olshansk/rss-feeds/main/feeds/feed_anthropic_engineering.xml", 
        ]
        self.converter= DocumentConverter()
    def get_articles(self, hours) -> List[AnthropicArticle]:
        now= datetime.now(timezone.utc)
        cutoff_time= now -timedelta(hours=hours)
        articles= []
        seen_guids=set()

        for rss_url in self.rss_urls:
            feed=feedparser.parse(rss_url)
            if not feed.entries:
                continue
            for entry in feed.entries:
                published_parsed= getattr(entry, "published_parsed", None)
                if not published_parsed:
                    continue
                published_time= datetime(*published_parsed[:6], tzinfo=timezone.utc)
                if published_time >= cutoff_time:
                    guid= entry.get("id", entry.get("link",""))
                    if guid not in seen_guids:
                        seen_guids.add(guid)
                        articles.append(AnthropicArticle(
                            title=entry.get("title", ""),
                            description=entry.get("description", ""),
                            url=entry.get("link", ""),
                            guid=guid,
                            published_at=published_time,
                            category=entry.get("tags", [{}])[0].get("term") if entry.get("tags") else None                           
                        ))
        return articles




    def url_to_markdown(self, url) -> Optional[str]:
        try:
            result= self.converter.convert(url)
            return result.document.export_to_markdown()

        except Exception:
            return None
        




scrapper= AnthropicScrapper()
articles: List[AnthropicArticle]= scrapper.get_articles(hours=100)
markdown: str = scrapper.url_to_markdown(articles[1].url)
print(markdown)

# Agents for financial services

May 5, 2026

Agents for financial services

<!-- image -->

We're releasing ten ready-to-run agent templates for the most time-consuming work in financial services: building pitchbooks, screening KYC files, and closing the books at month-end. Each one ships as a [plugin](https://support.claude.com/en/articles/13837440-use-plugins-in-claude-cowork) in Claude Cowork and Claude Code, and as a cookbook for [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) , so a team can put Claude on real financial work in days rather than months.

Claude also now works across Microsoft Excel, PowerPoint, Word, and Outlook (coming soon) through the Claude add-ins for Microsoft 365. Once the add-ins are installed, context carries automatically between applications, so work that starts in a model can end in a deck without re-explaining anything in between.

Finally, we're continuing to expand our partner ecosystem with new connectors and an